# L. monocytogenes Amino Acid Biosynthesis and Carbon Source Utilization

*Listeria monocytogenes* is a facultative intracellular pathogen found in diverse environments:
food processing, clinical settings, soil, water, and vegetation. It is a major foodborne
pathogen causing listeriosis, with well-characterized metabolic requirements.

This notebook uses **GapMind pathway predictions** from the BERDL pangenome database to ask:

1. Which amino acid biosynthesis pathways are complete vs incomplete across *L. monocytogenes* genomes?
2. Which carbon sources does the pangenome predict *L. monocytogenes* can use?
3. Do the three GTDB clades differ in metabolic pathway completeness?
4. Do pathway profiles vary across food, clinical, and environmental isolates?

**Key methodological difference from hardcoded approaches**: All pathway definitions come
from GapMind (pre-computed in the lakehouse), not from manually curated gene lists.
GapMind uses HMM profiles to assess pathway step completeness, providing traceable,
reproducible pathway scoring.

**Portability**: This notebook uses Spark SQL on BERDL JupyterHub.

## 1. Setup

In [ ]:
from berdl_notebook_utils import get_spark_session
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

spark = get_spark_session("ListeriaPathways")

## 2. Identify L. monocytogenes in the pangenome

GTDB splits *L. monocytogenes* into multiple species clades. We identify all of them
and their pangenome statistics.

In [ ]:
listeria_clades = spark.sql("""
    SELECT s.GTDB_species, s.gtdb_species_clade_id,
           p.no_genomes, p.no_core, p.no_gene_clusters,
           p.no_aux_genome, p.no_singleton_gene_clusters
    FROM kbase_ke_pangenome.pangenome p
    JOIN kbase_ke_pangenome.gtdb_species_clade s
      ON p.gtdb_species_clade_id = s.gtdb_species_clade_id
    WHERE s.GTDB_species LIKE '%Listeria_monocytogenes%'
    ORDER BY p.no_genomes DESC
""")

listeria_clades_pd = listeria_clades.toPandas()
print(f"L. monocytogenes clades in BERDL: {len(listeria_clades_pd)}")
print(f"Total genomes: {listeria_clades_pd['no_genomes'].sum()}")
listeria_clades_pd

## 3. GapMind amino acid pathway completeness

GapMind provides pre-computed pathway predictions for amino acid biosynthesis
and carbon source utilization. Each genome-pathway pair may have multiple rows
(one per step), so we take the best score per pair.

Score categories (ordered): `complete` > `likely_complete` > `steps_missing_low` >
`steps_missing_medium` > `not_present`.

See `docs/pitfalls.md` [pangenome_pathway_geography] for the multi-row pitfall.

In [ ]:
# Collect clade IDs for filtering
clade_ids = listeria_clades_pd['gtdb_species_clade_id'].tolist()
clade_filter = "','".join(clade_ids)

# Query GapMind for amino acid pathways, handling multi-row steps
aa_pathways = spark.sql(f"""
    WITH scored AS (
        SELECT
            clade_name,
            genome_id,
            pathway,
            metabolic_category,
            score,
            score_category,
            score_simplified,
            nHi, nMed, nLo,
            CASE score_category
                WHEN 'complete' THEN 5
                WHEN 'likely_complete' THEN 4
                WHEN 'steps_missing_low' THEN 3
                WHEN 'steps_missing_medium' THEN 2
                WHEN 'not_present' THEN 1
                ELSE 0
            END as score_value
        FROM kbase_ke_pangenome.gapmind_pathways
        WHERE clade_name IN ('{clade_filter}')
          AND metabolic_category = 'amino_acid'
    ),
    best_per_genome_pathway AS (
        SELECT
            clade_name,
            genome_id,
            pathway,
            MAX(score_value) as best_score,
            MAX(score) as best_raw_score,
            MAX(nHi) as max_nHi,
            MAX(nMed) as max_nMed,
            MAX(nLo) as max_nLo
        FROM scored
        GROUP BY clade_name, genome_id, pathway
    )
    SELECT *,
        CASE best_score
            WHEN 5 THEN 'complete'
            WHEN 4 THEN 'likely_complete'
            WHEN 3 THEN 'steps_missing_low'
            WHEN 2 THEN 'steps_missing_medium'
            WHEN 1 THEN 'not_present'
            ELSE 'unknown'
        END as best_category
    FROM best_per_genome_pathway
""")

aa_pd = aa_pathways.toPandas()
print(f"Amino acid pathway assessments: {len(aa_pd)} (genome x pathway pairs)")
print(f"Genomes: {aa_pd['genome_id'].nunique()}")
print(f"Pathways: {sorted(aa_pd['pathway'].unique())}")
print()

# Summary: fraction of genomes with each pathway complete or likely complete
aa_summary = aa_pd.groupby('pathway').agg(
    n_genomes=('genome_id', 'nunique'),
    frac_complete=('best_score', lambda x: (x >= 5).mean()),
    frac_likely=('best_score', lambda x: (x >= 4).mean()),
    frac_any=('best_score', lambda x: (x >= 3).mean()),
    mean_score=('best_raw_score', 'mean')
).reset_index().sort_values('frac_complete', ascending=False)

print("=== Amino Acid Pathway Completeness (all L. monocytogenes genomes) ===")
print(f"{'Pathway':<15} {'Complete':>10} {'Likely+':>10} {'Any steps':>10} {'Mean score':>12}")
print("-" * 60)
for _, row in aa_summary.iterrows():
    print(f"{row['pathway']:<15} {row['frac_complete']:>9.1%} {row['frac_likely']:>9.1%} "
          f"{row['frac_any']:>9.1%} {row['mean_score']:>11.2f}")

## 4. Carbon source utilization

GapMind also predicts which carbon sources an organism can use. This is relevant
for *L. monocytogenes* because carbon source utilization may differ between
food, clinical, and environmental isolates.

In [ ]:
# Query GapMind for carbon source pathways
carbon_pathways = spark.sql(f"""
    WITH scored AS (
        SELECT
            clade_name,
            genome_id,
            pathway,
            score,
            score_category,
            CASE score_category
                WHEN 'complete' THEN 5
                WHEN 'likely_complete' THEN 4
                WHEN 'steps_missing_low' THEN 3
                WHEN 'steps_missing_medium' THEN 2
                WHEN 'not_present' THEN 1
                ELSE 0
            END as score_value
        FROM kbase_ke_pangenome.gapmind_pathways
        WHERE clade_name IN ('{clade_filter}')
          AND metabolic_category = 'carbon'
    ),
    best_per_genome_pathway AS (
        SELECT
            clade_name,
            genome_id,
            pathway,
            MAX(score_value) as best_score,
            MAX(score) as best_raw_score
        FROM scored
        GROUP BY clade_name, genome_id, pathway
    )
    SELECT *,
        CASE best_score
            WHEN 5 THEN 'complete'
            WHEN 4 THEN 'likely_complete'
            WHEN 3 THEN 'steps_missing_low'
            WHEN 2 THEN 'steps_missing_medium'
            WHEN 1 THEN 'not_present'
            ELSE 'unknown'
        END as best_category
    FROM best_per_genome_pathway
""")

carbon_pd = carbon_pathways.toPandas()
print(f"Carbon source pathway assessments: {len(carbon_pd)} (genome x pathway pairs)")
print(f"Pathways: {carbon_pd['pathway'].nunique()}")
print()

# Summary
carbon_summary = carbon_pd.groupby('pathway').agg(
    n_genomes=('genome_id', 'nunique'),
    frac_complete=('best_score', lambda x: (x >= 5).mean()),
    frac_likely=('best_score', lambda x: (x >= 4).mean()),
    mean_score=('best_raw_score', 'mean')
).reset_index().sort_values('frac_complete', ascending=False)

print("=== Carbon Source Utilization (fraction of genomes with complete pathway) ===")
print(f"{'Pathway':<20} {'Complete':>10} {'Likely+':>10} {'Mean score':>12}")
print("-" * 55)
for _, row in carbon_summary.iterrows():
    print(f"{row['pathway']:<20} {row['frac_complete']:>9.1%} {row['frac_likely']:>9.1%} "
          f"{row['mean_score']:>11.2f}")

## 5. Cross-clade comparison

GTDB splits *L. monocytogenes* into multiple species clades. These roughly
correspond to classical lineages with distinct ecology: lineage I is primarily
clinical, lineage II is more environmental/food-associated, and lineage III
is animal-associated. Do they differ in metabolic capacity?

In [ ]:
# Combine amino acid and carbon data
aa_pd['metabolic_category'] = 'amino_acid'
carbon_pd['metabolic_category'] = 'carbon'
all_pathways = pd.concat([aa_pd, carbon_pd], ignore_index=True)

# Per-clade amino acid pathway completeness
print("=== Amino Acid Pathway Completeness by Clade ===")
print()

for clade in sorted(all_pathways['clade_name'].unique()):
    clade_data = all_pathways[
        (all_pathways['clade_name'] == clade) &
        (all_pathways['metabolic_category'] == 'amino_acid')
    ]
    n_genomes = clade_data['genome_id'].nunique()
    # Short clade name for display
    short_name = clade.split('--')[0].replace('s__', '')
    print(f"{short_name} ({n_genomes} genomes):")

    clade_summary = clade_data.groupby('pathway').agg(
        frac_complete=('best_score', lambda x: (x >= 5).mean()),
        frac_likely=('best_score', lambda x: (x >= 4).mean())
    ).sort_values('frac_complete', ascending=False)

    for pathway, row in clade_summary.iterrows():
        status = 'COMPLETE' if row['frac_complete'] > 0.9 else \
                 'VARIABLE' if row['frac_complete'] > 0.1 else \
                 'likely' if row['frac_likely'] > 0.5 else 'ABSENT/RARE'
        print(f"  {pathway:<15} {row['frac_complete']:>6.1%} complete  "
              f"{row['frac_likely']:>6.1%} likely+  [{status}]")
    print()

# Carbon source comparison: count complete pathways per genome per clade
carbon_complete = all_pathways[
    (all_pathways['metabolic_category'] == 'carbon') &
    (all_pathways['best_score'] >= 5)
].groupby(['clade_name', 'genome_id']).size().reset_index(name='n_complete_carbon')

print("=== Complete Carbon Source Pathways per Genome by Clade ===")
for clade in sorted(carbon_complete['clade_name'].unique()):
    subset = carbon_complete[carbon_complete['clade_name'] == clade]
    short_name = clade.split('--')[0].replace('s__', '')
    print(f"{short_name}: {subset['n_complete_carbon'].mean():.1f} +/- "
          f"{subset['n_complete_carbon'].std():.1f} "
          f"(range {subset['n_complete_carbon'].min()}-{subset['n_complete_carbon'].max()})")

## 6. Environment cross-reference

Join GapMind pathway scores with environment metadata from the
`gene_environment_association` project. Do food vs clinical vs
environmental isolates show different metabolic profiles?

In [ ]:
# Load environment metadata
env_meta = pd.read_csv('../../../projects/gene_environment_association/data/genome_env_metadata.csv')

# Filter to L. monocytogenes and non-'other' environments
listeria_env = env_meta[
    env_meta['gtdb_species_clade_id'].isin(clade_ids) &
    (env_meta['env_category'] != 'other')
].copy()

print(f"L. monocytogenes genomes with environment metadata: {len(listeria_env)}")
print(f"Environment categories:")
print(listeria_env['env_category'].value_counts().to_string())
print()

# Merge with pathway data
aa_with_env = aa_pd.merge(
    listeria_env[['genome_id', 'env_category']],
    on='genome_id', how='inner'
)

# Filter to environments with >= 10 genomes
env_counts = aa_with_env.groupby('env_category')['genome_id'].nunique()
qualifying_envs = env_counts[env_counts >= 10].index.tolist()
aa_with_env = aa_with_env[aa_with_env['env_category'].isin(qualifying_envs)]

print(f"Qualifying environments (>=10 genomes): {qualifying_envs}")
print()

# Amino acid completeness by environment
print("=== Amino Acid Pathway Completeness by Environment ===")
print(f"{'Pathway':<15}", end='')
for env in sorted(qualifying_envs):
    print(f" {env:>15}", end='')
print()
print("-" * (15 + 16 * len(qualifying_envs)))

for pathway in sorted(aa_pd['pathway'].unique()):
    print(f"{pathway:<15}", end='')
    for env in sorted(qualifying_envs):
        subset = aa_with_env[
            (aa_with_env['pathway'] == pathway) &
            (aa_with_env['env_category'] == env)
        ]
        if len(subset) > 0:
            frac = (subset['best_score'] >= 5).mean()
            print(f" {frac:>14.1%}", end='')
        else:
            print(f" {'n/a':>14}", end='')
    print()

## 7. Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: Amino acid pathway completeness heatmap by clade
ax1 = axes[0]
pivot_aa = aa_pd.pivot_table(
    index='pathway',
    columns='clade_name',
    values='best_score',
    aggfunc=lambda x: (x >= 5).mean()
)
# Shorten column names
pivot_aa.columns = [c.split('--')[0].replace('s__Listeria_', 'L. ') for c in pivot_aa.columns]
pivot_aa = pivot_aa.sort_index()

sns.heatmap(pivot_aa, annot=True, fmt='.0%', cmap='RdYlGn',
            vmin=0, vmax=1, ax=ax1, cbar_kws={'label': 'Fraction complete'})
ax1.set_title('Amino Acid Pathway Completeness\n(GapMind, by GTDB clade)')
ax1.set_xlabel('')
ax1.set_ylabel('Pathway')

# Right: Carbon source utilization - top 20 by overall completeness
ax2 = axes[1]
top_carbon = carbon_summary.head(20).copy()
colors = ['#4CAF50' if v >= 0.9 else '#FF9800' if v >= 0.5 else '#F44336'
          for v in top_carbon['frac_complete']]
ax2.barh(range(len(top_carbon)), top_carbon['frac_complete'],
         color=colors)
ax2.set_yticks(range(len(top_carbon)))
ax2.set_yticklabels(top_carbon['pathway'])
ax2.set_xlabel('Fraction of genomes with complete pathway')
ax2.set_title('Carbon Source Utilization\n(GapMind, top 20 pathways)')
ax2.set_xlim(0, 1.05)
ax2.invert_yaxis()

plt.tight_layout()
plt.savefig('../figures/pathway_completeness.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: figures/pathway_completeness.png")

## 8. Validation against known biology

*L. monocytogenes* has well-characterized growth requirements from minimal
medium studies. We compare GapMind predictions against published auxotrophies.

**Known amino acid requirements** (Premaratne et al., 1991; Phan-Thanh & Gormon, 1997;
Tsai & Hodgson, 2003):
- Branched-chain amino acids (isoleucine, leucine, valine) are required
- Cysteine, methionine, and arginine are required or growth-enhancing
- Histidine and tryptophan requirements vary by strain

**Known carbon sources** (Pine et al., 1989; Trivett & Meyer, 1971):
- Glucose is the preferred carbon source
- Can use glycerol, mannose, cellobiose, trehalose
- Cannot use most sugar alcohols (except glycerol)

In [ ]:
# Validation: compare GapMind predictions against known biology
print("=== Validation: GapMind vs Known L. monocytogenes Biology ===")
print()

# Known amino acid auxotrophies (from literature)
# These are amino acids typically required in minimal media
known_required = ['ile', 'leu', 'val', 'cys', 'met', 'arg']
known_variable = ['his', 'trp']  # strain-dependent
known_synthesized = ['ala', 'gly', 'ser', 'thr', 'pro', 'asn', 'gln',
                     'phe', 'tyr', 'lys', 'chorismate']

print("Amino acid biosynthesis predictions vs known requirements:")
print(f"{'Pathway':<15} {'GapMind complete':>16} {'Expected':>15} {'Match?':>8}")
print("-" * 58)

for _, row in aa_summary.iterrows():
    pathway = row['pathway']
    frac = row['frac_complete']

    if pathway in known_required:
        expected = 'AUXOTROPHIC'
        match = 'YES' if frac < 0.5 else 'NO'
    elif pathway in known_variable:
        expected = 'variable'
        match = 'YES' if 0.1 < frac < 0.9 else '~'
    elif pathway in known_synthesized:
        expected = 'synthesized'
        match = 'YES' if frac > 0.5 else 'NO'
    else:
        expected = '?'
        match = '?'

    print(f"{pathway:<15} {frac:>15.1%} {expected:>15} {match:>8}")

print()
print("Note: GapMind pathway names may not map 1:1 to amino acid abbreviations.")
print("'chorismate' is the precursor for aromatic amino acids (phe, tyr, trp).")

# Carbon source validation
print()
print("Carbon source utilization predictions vs known biology:")
known_carbon_yes = ['glucose', 'glycerol', 'mannose', 'cellobiose', 'trehalose']
known_carbon_no = ['sorbitol', 'mannitol', 'lactose', 'xylose', 'rhamnose']

for pathway_name in known_carbon_yes + known_carbon_no:
    row = carbon_summary[carbon_summary['pathway'] == pathway_name]
    if len(row) > 0:
        frac = row.iloc[0]['frac_complete']
        expected = 'CAN USE' if pathway_name in known_carbon_yes else 'CANNOT USE'
        predicted = 'complete' if frac > 0.5 else 'incomplete'
        match = 'YES' if (frac > 0.5) == (pathway_name in known_carbon_yes) else 'NO'
        print(f"  {pathway_name:<15} GapMind: {frac:>5.1%} complete  "
              f"Expected: {expected:<12} {match}")
    else:
        print(f"  {pathway_name:<15} not in GapMind")

## 9. Summary

In [ ]:
n_genomes = aa_pd['genome_id'].nunique()
n_clades = aa_pd['clade_name'].nunique()
n_aa_pathways = aa_pd['pathway'].nunique()
n_carbon_pathways = carbon_pd['pathway'].nunique()

# Count universally complete AA pathways
universal_complete = aa_summary[aa_summary['frac_complete'] > 0.95]['pathway'].tolist()
universal_absent = aa_summary[aa_summary['frac_complete'] < 0.05]['pathway'].tolist()
variable_pathways = aa_summary[
    (aa_summary['frac_complete'] >= 0.05) & (aa_summary['frac_complete'] <= 0.95)
]['pathway'].tolist()

print(f"=== L. monocytogenes GapMind Pathway Analysis ===")
print(f"Genomes analyzed: {n_genomes} across {n_clades} GTDB clades")
print(f"Amino acid pathways: {n_aa_pathways}")
print(f"Carbon source pathways: {n_carbon_pathways}")
print()
print(f"Universally complete AA pathways (>95% of genomes): {universal_complete}")
print(f"Universally absent AA pathways (<5% of genomes): {universal_absent}")
print(f"Variable AA pathways (5-95%): {variable_pathways}")
print()

# Carbon source summary
carbon_complete_list = carbon_summary[carbon_summary['frac_complete'] > 0.5]['pathway'].tolist()
print(f"Carbon sources with >50% complete: {carbon_complete_list}")
print()
print("See RESEARCH_PLAN.md for hypothesis evaluation and validation context.")